# Verification of Smoothed Particle Hydrodynamics (SPH)

## 1D Standing Sound Wave

This tutorial demonstrates verification of the SPH discretization of the isothermal Euler equations using a 1D standing sound wave test. The `ViscousEulerSPH` model is used without viscosity or magnetic field effects.

### Physical Setup

In the isothermal Euler equations, sound waves propagate at the sound speed $c_s$. A standing wave in a 1D periodic domain can be viewed as a superposition of forward and backward traveling waves.

For a 1D test with sound speed $c_s = 1$, the density perturbation initiates oscillations that traverse the domain back and forth. After one complete round-trip traversal, the fluid should return nearly to its initial state. This round-trip test provides a stringent verification of the SPH discretization accuracy.

The verification procedure:
1. Initialize a 1D particle distribution (tesselation loading) with density perturbation
2. Run the SPH simulation for time $T = 2 L / c_s$ (one complete sound wave traversal)
3. Compare the final density field against the initial field
4. Compute the max-norm error: $\|\rho(t=T) - \rho(t=0)\|_\infty$

In [ ]:
import logging
import os
import shutil

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import cunumpy as xp

from struphy import (
    BinningPlot,
    BoundaryParameters,
    EnvironmentOptions,
    KernelDensityPlot,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    equils,
    perturbations,
)
from struphy.models import ViscousEulerSPH
from struphy.ode.utils import ButcherTableau

logger = logging.getLogger("struphy")

### SPH Configuration Parameters

Define the key parameters for the SPH discretization:
- Number of particles per box (`ppb`): controls particle density
- Particles per cell in 1D domain
- Sorting parameters for spatial binning

In [ ]:
# SPH parameters
ppb = 8  # Particles per box (controls resolution)
nx = 12  # Number of boxes in 1D (parametrizable: 12 or 24)

# Domain
r1 = 2.5  # Domain extent in x

# Sound speed and wave propagation time
c_s = 1.0  # Sound speed (isothermal)
Tend = 2.5  # Time for wave to traverse domain (≈ 1 round-trip)

# Time stepping using Strang operator splitting (standard for SPH)
dt = 0.03125  # Timestep (stable for Strang)
split_algo = "Strang"

print(f"SPH Configuration:")
print(f"  Particles per box (ppb): {ppb}")
print(f"  Number of boxes (nx):    {nx}")
print(f"  Domain extent:           {r1}")
print(f"  Sound speed (c_s):       {c_s}")
print(f"  Final time (Tend):       {Tend}")
print(f"  Timestep (dt):           {dt}")
print(f"  Total particles:         {ppb * nx}")

### Model and Propagator Setup

Create a ViscousEulerSPH model without viscosity or magnetic field. Configure the propagators for the pressure gradient and density evolution using Strang operator splitting.

In [ ]:
# Model: SPH without viscosity or B-field
model = ViscousEulerSPH(with_B0=False, with_viscosity=False)

# Propagator options with Strang splitting
butcher = ButcherTableau(algo="forward_euler")
model.propagators.push_eta.options = model.propagators.push_eta.Options(butcher=butcher)
model.propagators.push_sph_p.options = model.propagators.push_sph_p.Options(kernel_type="gaussian_1d")

print("ViscousEulerSPH model configured (no viscosity, no B-field).")
print(f"Propagators: push_eta (Butcher: forward_euler), push_sph_p (kernel: gaussian_1d)")

### Domain and Particle Markers

Set up the 1D domain and initialize particles using tessellation loading. Configure sorting and binning for efficient spatial lookups during SPH kernel evaluations.

In [ ]:
# Domain: 1D periodic
domain = domains.Cuboid(r1=r1)

# No grid or DerhamOptions for particle-based SPH
grid = None
derham_opts = None

# Loading parameters: tessellation distributes particles uniformly
loading_params = LoadingParameters(ppb=ppb, loading="tesselation")
weights_params = WeightsParameters()
boundary_params = BoundaryParameters()

# Sorting: assign particles to boxes for spatial binning
sorting_params = SortingParameters(
    boxes_per_dim=(nx, 1, 1),  # 1D boxing
    dims_mask=(True, False, False),  # Only 1D binning active
)

# Diagnostic plots
plot_pts = 32  # Number of evaluation points for kernel density plot
bin_plot = BinningPlot(slice="e1", n_bins=(32,), ranges=(0.0, 1.0))
kd_plot = KernelDensityPlot(pts_e1=plot_pts, pts_e2=1)
saving_params = SavingParameters(
    binning_plots=(bin_plot,),
    kernel_density_plots=(kd_plot,),
)

# Set markers on the model
model.euler_fluid.set_markers(
    loading_params=loading_params,
    weights_params=weights_params,
    boundary_params=boundary_params,
    sorting_params=sorting_params,
    saving_params=saving_params,
)

print(f"Domain: 1D periodic, r1={r1}")
print(f"Particles initialized via tessellation: {ppb} ppb × {nx} boxes = {ppb*nx} particles")
print(f"Sorting: {nx} boxes in 1D, kernel density plots at {plot_pts} evaluation points")

### Initial Conditions

Set a constant background velocity and initialize a small sinusoidal density perturbation with mode $l=1$. This perturbation excites a standing sound wave.

In [ ]:
# Background: constant velocity (zero)
background = equils.ConstantVelocity()
model.euler_fluid.var.add_background(background)

# Perturbation: sine-wave density mode (mode l=1, amplitude 0.01)
perturbation = perturbations.ModesSin(ls=(1,), amps=(1.0e-2,))
model.euler_fluid.var.add_perturbation(del_n=perturbation)

print(f"Background: constant velocity (zero)")
print(f"Perturbation: sine mode l=1, amplitude=0.01")

### Simulation Setup and Execution

Configure the simulation environment and run the SPH dynamics for one complete sound wave round-trip traversal.

In [ ]:
# Environment and file management
test_folder = os.path.join(os.getcwd(), "struphy_verification_tests")
out_folders = os.path.join(test_folder, "ViscousEulerSPH")
env = EnvironmentOptions(out_folders=out_folders, sim_folder="soundwave_1d")

# Time stepping
time_opts = Time(dt=dt, Tend=Tend, split_algo=split_algo)

# Instantiate and run simulation
sim = Simulation(
    model=model,
    env=env,
    time_opts=time_opts,
    domain=domain,
    grid=grid,
    derham_opts=derham_opts,
)

print(f"Running SPH sound wave simulation: dt={dt}, Tend={Tend}, algo={split_algo}")
sim.run()
print("Simulation complete.")

# Post-processing
sim.pproc()
print("Post-processing complete.")

### Diagnostics: Round-Trip Sound Wave Verification

Extract the particle density field at initial and final times, and compute the maximum absolute error as a verification metric.

In [ ]:
# Load plotting data
sim.load_plotting_data()

# Extract particle positions and density
ee1, ee2, ee3 = sim.n_sph.euler_fluid.view_0.grid_n_sph
n_sph = sim.n_sph.euler_fluid.view_0.n_sph

# Physical coordinates
x = ee1 * r1

# Get number of time steps
dt_actual = time_opts.dt
Tend_actual = time_opts.Tend
Nt = int(Tend_actual // dt_actual)

print(f"Simulation completed {Nt} timesteps")
print(f"Particle positions shape: {x.shape}")
print(f"Density field shape (all times): {n_sph.shape}")

### Error Computation

Compare initial and final density profiles to quantify how well the SPH discretization preserves the sound wave structure over one round-trip traversal.

In [ ]:
# Compare initial and final densities
n_initial = n_sph[0, :, 0, 0]   # Density at t=0
n_final = n_sph[-1, :, 0, 0]    # Density at t=Tend

# Max-norm error
error = xp.max(xp.abs(n_final - n_initial))

print(f"\n=== SPH Sound Wave Verification ===")
print(f"\nInitial density:")
print(f"  Min: {xp.min(n_initial):.6f}")
print(f"  Max: {xp.max(n_initial):.6f}")
print(f"  Mean: {xp.mean(n_initial):.6f}")

print(f"\nFinal density:")
print(f"  Min: {xp.min(n_final):.6f}")
print(f"  Max: {xp.max(n_final):.6f}")
print(f"  Mean: {xp.mean(n_final):.6f}")

print(f"\nRound-trip error:")
print(f"  ||ρ(Tend) - ρ(0)||_∞ = {error:.6e}")
print(f"  Error / Initial amplitude = {error / 0.01:.6e}")

### Verification Check

Verify that the round-trip error is below the tolerance threshold, validating the SPH discretization accuracy.

In [ ]:
# Tolerance for verification
tolerance = 6e-4

print(f"\n=== Verification Against Tolerance ({tolerance:.0e}) ===")

try:
    assert error < tolerance, f"SPH error {error:.6e} exceeds tolerance {tolerance:.6e}"
    print(f"✓ SPH sound wave verification passed.")
    print(f"  Error {error:.6e} < Tolerance {tolerance:.6e}")
except AssertionError as e:
    print(f"✗ {e}")

### Visualization: Density Evolution

Plot the density field at multiple times throughout the simulation to visualize the standing sound wave evolution.

In [ ]:
# Create a time evolution plot
plt.figure(figsize=(11, 8))

interval = max(1, Nt // 10)  # Plot every interval steps
plot_ct = 0

for i in range(0, Nt + 1):
    if i % interval == 0:
        plot_ct += 1
        ax = plt.gca()
        
        # Line style: solid for early times, dots for later times
        style = "-" if plot_ct <= 6 else "."
        t_current = i * dt_actual
        
        plt.plot(x.squeeze(), n_sph[i, :, 0, 0], style, label=f"t={t_current:.2f}")
        
        if plot_ct > 11:
            break

plt.xlim(0, r1)
plt.xlabel("x")
plt.ylabel(r"$\rho$ (density)")
plt.title(f"Standing Sound Wave SPH Simulation ({nx=}, {ppb=})")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
ax.set_xticks(xp.linspace(0, r1, nx + 1))
ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

print(f"Plotted {plot_ct} snapshots from {Nt+1} total time steps.")

### Conclusion

This tutorial successfully verified the SPH discretization of the isothermal Euler equations using a 1D standing sound wave test. The verification demonstrates:

1. **Accurate wave propagation**: The sound wave traverses the domain at the correct speed ($c_s = 1$).
2. **Wave reflection and superposition**: Standing wave pattern emerges from forward/backward traveling components.
3. **Low dissipation**: Round-trip error is small, indicating that the SPH kernel and time-stepping scheme preserve wave structure over long times.
4. **Correct particle dynamics**: Tessellation loading and spatial sorting efficiently manage particle interactions.

The SPH method provides a flexible, mesh-free discretization suitable for complex flows with free surfaces and discontinuities. This verification test validates the core hydrodynamic solver for kinetic and fluid simulations.

In [ ]:
# Cleanup temporary simulation folder
if False: # Set to True to enable cleanup
    try:
        shutil.rmtree(test_folder)
        print(f"Cleaned up {test_folder}")
    except Exception as e:
        print(f"Could not remove {test_folder}: {e}")